# KBM GFTM accuracy vs FILTER and WIDTH (Fusion_PhD-38e9.3)

Advisory. Dominant mode = argmax growth rate over `mode` among unstable modes (never `isel(mode=0)`).
Reference = GS2 tail-averaged `pyro_cube_avg`. Population = GS2 growth rate finite and > 0, identical in every cell.
**Error metric (same as `kbm_extent_filter` fig5): medlog = median |log10(g_GFTM/g_GS2)|, medAE = median |g_GFTM - g_GS2|,
and median |omega_GFTM - omega_GS2| for frequency.** A case with no unstable mode is a MISS: g = 0 and omega = 0, so
its log error is +inf (it stays in the median, and misses are counted and plotted, never dropped).
Everything else is the `wo_*` family: NBASIS_MAX 22, NXGRID 28, NMODES 4, IBRANCH -1, FIND_WIDTH F.
FILTER 0 = `wo_<w>`, 0.5 = `wo_f05_<w>`, the others = `wo_fNN_<w>` (GyroRun, `pyroscan.nc`).

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
import pyrokinetics
from pyrokinetics.pyroscan import PyroScanGKOutput

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir())
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
load_dotenv(ROOT / "local.env", override=True)
print("pyrokinetics", pyrokinetics.__version__)

analysis_name = "kbm_filter_width"
data_root = Path(os.environ["GK_DATA_ROOT"]).expanduser()
project = "LATIN_HYPERCUBE"
cases = {"SPR-045": "SPR-045", "M1": "M1"}   # database label -> case directory
widths = ["w0p2000", "w0p2297", "w0p2639", "w0p3031", "w0p3482", "w0p4000", "w0p4595",
          "w0p5279", "w0p6063", "w0p6965", "w0p8000", "w0p9000", "w1p0000", "w1p1000",
          "w1p2000", "w1p3000", "w1p4000", "w1p5000", "w1p6000", "w1p9218", "w2p3083",
          "w2p7726", "w3p3302", "w4p0000"]
width_value = {w: float(w[1:].replace("p", ".")) for w in widths}
filters = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

def leaf(filt, w):
    # FILTER 0 and 0.5 pre-date GyroRun and keep the old cube layout; the rest are GyroRun leaves
    if filt == 0.0:
        return f"kbm_8d/wo_{w}/pyro_cube/cube.nc"
    if filt == 0.5:
        return f"kbm_8d/wo_f05_{w}/pyro_cube/cube.nc"
    return f"kbm_8d/wo_f{round(filt * 10):02d}_{w}/pyroscan.nc"

In [ ]:
def mag(v):
    return np.asarray(getattr(v.data, "magnitude", v.data))

def select(gam):
    # argmax growth over 'mode' among unstable modes; 0 where none is unstable (a MISS)
    g = np.where(gam > 0, gam, -np.inf)
    idx = g.argmax(axis=1)
    return np.where(np.isfinite(g.max(axis=1)), gam[np.arange(len(gam)), idx], 0.0), idx

gs2 = {db: PyroScanGKOutput.from_netcdf(data_root / "GS2" / "Runs" / project / case / "kbm_8d/pyro_cube_avg/cube.nc").data
       for db, case in cases.items()}
ref = {}
for db, d in gs2.items():
    g, f = mag(d.growth_rate), mag(d.mode_frequency)
    ok = np.isfinite(g) & (g > 0)
    ref[db] = dict(names=d.sample_name.values[ok], g=g[ok], f=f[ok], gu=str(d.growth_rate.data.units), fu=str(d.mode_frequency.data.units))
    assert np.isfinite(ref[db]["f"]).all()
    print(db, "population", ok.sum(), "of", len(g))

In [ ]:
rows, caselog, missing = [], {}, []
for db, case in cases.items():
    r = ref[db]
    for filt in filters:
        for w in widths:
            p = data_root / "GFTM" / "Runs" / project / case / leaf(filt, w)
            if not p.is_file():
                missing.append((db, filt, w)); continue
            d = PyroScanGKOutput.from_netcdf(p).data
            assert str(d.growth_rate.data.units) == r["gu"] and str(d.mode_frequency.data.units) == r["fu"], "normalisations differ"
            idx = pd.Index(d.sample_name.values).get_indexer(r["names"])
            assert (idx >= 0).all(), f"{db}/{filt}/{w}: population cases missing"
            gam = mag(d.growth_rate.transpose("sample", "mode"))[idx]
            fr = mag(d.mode_frequency.transpose("sample", "mode"))[idx]
            assert np.isfinite(fr[gam > 0]).all()
            g_sel, i_sel = select(gam)
            f_sel = np.where(g_sel > 0, fr[np.arange(len(gam)), i_sel], 0.0)
            with np.errstate(divide="ignore"):
                logerr = np.abs(np.log10(g_sel / r["g"]))
            caselog[db, filt, width_value[w]] = logerr
            rows.append(dict(db=db, filter=filt, width=width_value[w], n=len(g_sel),
                             medlog=np.median(logerr), medAE=np.median(np.abs(g_sel - r["g"])),
                             medAE_freq=np.median(np.abs(f_sel - r["f"])), miss=int((g_sel == 0).sum())))
res = pd.DataFrame(rows)
print("leaves scored:", len(res), "| missing:", len(missing))
print(pd.Series([m[:2] for m in missing]).value_counts().to_string() if missing else "no missing leaves")

In [ ]:
# FILTER=0 and 0.5 are reused from earlier runs: check they reproduce the kbm_extent_filter fig5 numbers' structure
print(res.groupby(["db", "filter"]).agg(n_widths=("width", "size"), min_medlog=("medlog", "min"), min_medAE=("medAE", "min"),
                                        max_miss=("miss", "max")).round(4).to_string())

In [ ]:
cmap = plt.get_cmap("viridis")
rows_spec = [("medAE_freq", r"LogMed $|\omega_{\rm GFTM}-\omega_{\rm GS2}|$"),
             ("medAE", r"LogMed $|\gamma_{\rm GFTM}-\gamma_{\rm GS2}|$"),
             ("miss", "Misses")]
from matplotlib.ticker import StrMethodFormatter

def lines(xkey, ckey, xs, cs, fmt, xlabel, clabel, logx):
    # user's format (kbm_extent_filter fig 9): y = log10(median abs error), log WIDTH axis with fixed ticks, fonts 16/14/12
    fig, axes = plt.subplots(len(rows_spec), len(cases), figsize=(13, 12), sharex=True, squeeze=False)
    for col, db in enumerate(cases):
        for row, (m, lab) in enumerate(rows_spec):
            ax = axes[row, col]
            for i, c in enumerate(cs):
                s = res[(res.db == db) & (res[ckey] == c)].sort_values(xkey)
                y = s[m] if m == "miss" else np.log10(s[m])
                ax.plot(s[xkey], y, color=cmap(i / max(len(cs) - 1, 1)), marker="o", ms=2.5, lw=1)
            ax.set_ylabel(lab, fontsize=16)
            ax.tick_params(axis="both", labelsize=14)
            if logx:
                xl = ax.get_xlim()
                ax.set_xscale("log"); ax.set_xticks([0.5, 1, 2, 5, 10]); ax.set_xlim(*xl)
                ax.xaxis.set_major_formatter(StrMethodFormatter("{x:g}")); ax.minorticks_off()
        axes[0, col].set_title(db, fontsize=16)
        axes[-1, col].set_xlabel(xlabel, fontsize=16)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, len(cs) - 1))
    cb = fig.colorbar(sm, ax=axes, ticks=range(len(cs)), pad=0.02)
    cb.ax.set_yticklabels([fmt(c) for c in cs]); cb.set_label(clabel, fontsize=16); cb.ax.tick_params(labelsize=12)
    return fig

fig1 = lines("filter", "width", filters, sorted(res.width.unique()), lambda c: f"{c:.3g}", "FILTER", "WIDTH", False)
plt.show()

In [ ]:
fig2 = lines("width", "filter", widths, filters, lambda c: f"{c:.1f}", "WIDTH", "FILTER", True)
plt.show()

In [ ]:
# best (WIDTH, FILTER) per database, for each metric; ties broken toward fewer misses then lower medlog
best = []
for db in cases:
    s = res[res.db == db]
    for m in ("medlog", "medAE", "medAE_freq"):
        b = s.sort_values([m, "miss"]).iloc[0]
        best.append(dict(db=db, metric=m, width=b.width, filter=b["filter"], value=b[m], medlog=b.medlog, medAE=b.medAE,
                         medAE_freq=b.medAE_freq, miss=int(b.miss), n=int(b.n)))
best = pd.DataFrame(best)
print(best.round(4).to_string(index=False))

# best FILTER at each width, and best WIDTH at each FILTER (medlog)
for db in cases:
    s = res[res.db == db]
    bf = s.loc[s.groupby("width").medlog.idxmin()][["width", "filter", "medlog", "miss"]]
    print(db, "best FILTER per WIDTH (medlog):"); print(bf.round(4).to_string(index=False))
    bw = s.loc[s.groupby("filter").medlog.idxmin()][["filter", "width", "medlog", "miss"]]
    print(db, "best WIDTH per FILTER (medlog):"); print(bw.round(4).to_string(index=False))

In [ ]:
# per-case oracle over (WIDTH, FILTER) of the dominant-mode error: a BOUND that consults GS2, like the width oracle
summ = []
for db in cases:
    keys = [k for k in caselog if k[0] == db]
    stack = np.array([caselog[k] for k in keys])            # (n_settings, n_cases)
    bestk = stack.argmin(axis=0)
    ofilt = np.array([keys[i][1] for i in bestk]); owid = np.array([keys[i][2] for i in bestk])
    err = stack.min(axis=0)
    summ.append(dict(db=db, n=stack.shape[1], oracle_medlog=np.median(err), unreachable=int(np.isinf(err).sum()),
                     oracle_filter_median=np.median(ofilt), oracle_width_median=np.median(owid)))
    print(db, "share of cases whose oracle FILTER is ...")
    print(pd.Series(ofilt).value_counts(normalize=True).sort_index().round(3).to_string())
print(pd.DataFrame(summ).round(4).to_string(index=False))

In [ ]:
out = ROOT / "Plots" / analysis_name
out.mkdir(parents=True, exist_ok=True)
fig1.savefig(out / "fig1_errors_vs_filter.png")
fig2.savefig(out / "fig2_errors_vs_width.png")